In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "run.py").is_file())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
# All preparation paths are local to this notebook.
# For files fetched by the Download notebook, use PROJECT_ROOT / "artifacts/raw".
RAW_ROOT = PROJECT_ROOT / "datasets/raw"
ASSET_ROOT = PROJECT_ROOT / "notebooks/assets"
PROCESSED_ROOT = PROJECT_ROOT / "artifacts/data"


# DC_Flow
Fixed settings and data paths are defined in this notebook; processing functions live in `data/cities/`.


In [ ]:
from data.cities.dc import process_flow as process
OPTIONS = {'year': 2025,
 'freq': '60min',
 'flow_mode': 'arrival',
 'start': '2025-01-01 00:00:00',
 'end': '2026-01-01 00:00:00',
 'horizons': [1, 3, 6, 9, 12],
 'target_clusters': None,
 'target_tag': 'subset'}
result = process(OPTIONS, raw_root=RAW_ROOT, asset_root=ASSET_ROOT, processed_root=PROCESSED_ROOT)
# Expose returned arrays only in this interactive notebook for the plots below.
globals().update(result)

import numpy as np
import matplotlib.pyplot as plt


In [ ]:
MOBILITY_VIZ = [("taxi", dc_taxi_nt, STEPS), ("bike", dc_bike_nt, STEPS)]

def plot_cluster_heat(gdf, values, title, cmap="YlOrRd", log=True):
    plot_gdf = gdf.copy()
    vals = np.asarray(values, dtype=np.float64)
    plot_gdf["flow"] = np.log1p(vals) if log else vals
    legend_label = "log1p(total flow)" if log else "total flow"
    fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    plot_gdf.plot(column="flow", cmap=cmap, linewidth=0.2, edgecolor="black",
                  legend=True, legend_kwds={"label": legend_label, "shrink": 0.6}, ax=ax)
    ax.set_title(title)
    ax.set_axis_off()
    plt.tight_layout()
    plt.show()

def sparsity_report(name, arr, time_index):
    arr = np.asarray(arr)
    total_cells = arr.size
    nnz = int(np.count_nonzero(arr))
    zero_rate = 1.0 - nnz / total_cells if total_cells else 0.0
    total_flow = float(arr.sum())
    region_sum = arr.sum(axis=1)
    step_sum = arr.sum(axis=0)
    dead_regions = int((region_sum == 0).sum())
    dead_steps = int((step_sum == 0).sum())
    region_q = np.quantile(region_sum, [0.0, 0.25, 0.5, 0.75, 1.0])
    step_q = np.quantile(step_sum, [0.0, 0.25, 0.5, 0.75, 1.0])
    nz_step_q = np.quantile(step_sum[step_sum > 0], [0.25, 0.5, 0.75]) if (step_sum > 0).any() else np.array([0, 0, 0])
    print(f"=== Sparsity report: {name} ===")
    print(f"  shape={arr.shape}, total_flow={total_flow:,.0f}")
    print(f"  nnz cells={nnz:,}/{total_cells:,}  zero_rate={zero_rate*100:.3f}%")
    print(f"  dead regions={dead_regions}/{arr.shape[0]}  dead steps={dead_steps}/{arr.shape[1]}")
    print(f"  region totals  min/Q1/med/Q3/max = {region_q[0]:,.0f} / {region_q[1]:,.0f} / {region_q[2]:,.0f} / {region_q[3]:,.0f} / {region_q[4]:,.0f}")
    print(f"  step totals    min/Q1/med/Q3/max = {step_q[0]:,.0f} / {step_q[1]:,.0f} / {step_q[2]:,.0f} / {step_q[3]:,.0f} / {step_q[4]:,.0f}")
    print(f"  step totals  (non-zero only) Q1/med/Q3 = {nz_step_q[0]:,.0f} / {nz_step_q[1]:,.0f} / {nz_step_q[2]:,.0f}")
    idx_active = np.where(step_sum > 0)[0]
    if idx_active.size:
        print(f"  first_active={pd.Timestamp(time_index[idx_active[0]])}, last_active={pd.Timestamp(time_index[idx_active[-1]])}")

def plot_temporal_panels(name, arr, time_index):
    arr = np.asarray(arr)
    ts = pd.to_datetime(time_index)
    series = pd.Series(arr.sum(axis=0), index=ts)
    daily = series.resample("D").sum()
    hod = pd.DataFrame({"flow": series.values}, index=ts)
    hod["dow"] = hod.index.dayofweek
    hod["hour"] = hod.index.hour
    pivot = hod.pivot_table(index="dow", columns="hour", values="flow", aggfunc="sum").reindex(index=range(7), columns=range(24))
    fig, axes = plt.subplots(1, 2, figsize=(15, 4))
    axes[0].plot(daily.index, daily.values, linewidth=1.2)
    axes[0].set_title(f"{name} - daily total flow")
    axes[0].set_xlabel("Date")
    axes[0].set_ylabel("Daily total flow")
    axes[0].grid(alpha=0.25)
    im = axes[1].imshow(pivot.values, aspect="auto", cmap="viridis", origin="lower")
    axes[1].set_title(f"{name} - weekday x hour total flow")
    axes[1].set_xlabel("Hour of day")
    axes[1].set_ylabel("Day of week (0=Mon)")
    axes[1].set_xticks(range(0, 24, 3))
    axes[1].set_yticks(range(7))
    axes[1].set_yticklabels(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])
    fig.colorbar(im, ax=axes[1], shrink=0.8, label="flow")
    plt.tight_layout()
    plt.show()

In [ ]:
cluster_active = TARGET_CLUSTERS is not None and (not isinstance(TARGET_CLUSTERS, str) or TARGET_CLUSTERS.lower() != "all")
for name, arr, time_index in MOBILITY_VIZ:
    sparsity_report(name, arr, time_index)
    plot_cluster_heat(clusters, arr.sum(axis=1), f"DC {name} total flow {YEAR} ({FLOW_MODE}) - all clusters")
    if cluster_active:
        sub = arr[sel_idx]
        sparsity_report(f"{name} [{TARGET_TAG}]", sub, time_index)
        plot_cluster_heat(clusters_sel, sub.sum(axis=1), f"DC {name} total flow {YEAR} ({FLOW_MODE}) - {TARGET_TAG}")
    plot_temporal_panels(name, arr, time_index)